In [196]:
import random
from datetime import datetime, timedelta
import pandas as pd

In [197]:
# Set random seed for reproducibility
random.seed(42)

In [198]:
# Configuration parameters
NUM_PATIENTS = 200
start_date = datetime(2026, 6, 1, 8, 0, 0)

In [199]:
# Reference data pools
genders = ["Male", "Female", "Other"]
doctors = ["Dr. Robert", "Dr. House", "Dr. Smith", "Dr. Adams", "Dr. Chen"]
triage_levels = ["ESI Level 2", "ESI Level 3", "ESI Level 4", "ESI Level 5"]

In [200]:
# Locations mapped to activities
activity_locations = {
    "Check-in": "Reception Desk",
    "Triage": "Triage Room 1",
    "Doctor Consultation": "Consultation Bay B",
    "X-Ray": "Radiology Lab 2",
    "Discharge": "Exit Lobby",
}

In [201]:
resources = {
    "Check-in": "Front Desk Staff",
    "Triage": "Nurse Sarah",
    "Doctor Consultation": "Attending Physician",
    "X-Ray": "Tech Mike",
    "Discharge": "Discharge Coordinator",
}

In [202]:
# Reset event rows
event_rows = []

# Consolidated loop to generate all 200 sequential patient records (1000 to 1199)
for i in range(NUM_PATIENTS):
    case_id = 1000 + i
    patient_age = random.randint(18, 85)
    patient_gender = random.choice(genders)
    attending_doctor = random.choice(doctors)
    acuity_level = random.choice(triage_levels)

    # 40% chance that a patient is missing paperwork, triggering the rework loop after X-Ray
    paperwork_missing_flag = random.random() < 0.40

    # Stagger patient arrival times
    current_time = start_date + timedelta(
        minutes=random.randint(0, 1440 * 3)
    )  # Spread over 3 days

    # 1. Check-in
    event_rows.append({
        "case_id": case_id,
        "activity": "Check-in",
        "timestamp": current_time,
        "resource": resources["Check-in"],
        "location": activity_locations["Check-in"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 2. Triage
    current_time += timedelta(minutes=random.randint(5, 20))
    event_rows.append({
        "case_id": case_id,
        "activity": "Triage",
        "timestamp": current_time,
        "resource": resources["Triage"],
        "location": activity_locations["Triage"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 3. Doctor Consultation
    current_time += timedelta(minutes=random.randint(15, 60))
    event_rows.append({
        "case_id": case_id,
        "activity": "Doctor Consultation",
        "timestamp": current_time,
        "resource": attending_doctor,
        "location": activity_locations["Doctor Consultation"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 4. X-Ray
    current_time += timedelta(minutes=random.randint(10, 30))
    event_rows.append({
        "case_id": case_id,
        "activity": "X-Ray",
        "timestamp": current_time,
        "resource": resources["X-Ray"],
        "location": activity_locations["X-Ray"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # --- INJECTED BOTTLENECK / REWORK LOOP ---
    if paperwork_missing_flag:
        # Sent back to Triage due to missing form
        current_time += timedelta(minutes=random.randint(15, 30))
        event_rows.append({
            "case_id": case_id,
            "activity": "Triage",  # Rework step
            "timestamp": current_time,
            "resource": resources["Triage"],
            "location": activity_locations["Triage"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": True,
        })

        # Quick re-consultation after form fix
        current_time += timedelta(minutes=random.randint(10, 25))
        event_rows.append({
            "case_id": case_id,
            "activity": "Doctor Consultation",
            "timestamp": current_time,
            "resource": attending_doctor,
            "location": activity_locations["Doctor Consultation"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": False,
        })

    # 5. Discharge
    current_time += timedelta(minutes=random.randint(15, 40))
    event_rows.append({
        "case_id": case_id,
        "activity": "Discharge",
        "timestamp": current_time,
        "resource": resources["Discharge"],
        "location": activity_locations["Discharge"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

In [203]:
# Convert to DataFrame and sort before printing
df_events = pd.DataFrame(event_rows)
df_events = df_events.sort_values(by=["case_id", "timestamp"]).reset_index(drop=True)

print(df_events['case_id'])

0       1000
1       1000
2       1000
3       1000
4       1000
        ... 
1137    1199
1138    1199
1139    1199
1140    1199
1141    1199
Name: case_id, Length: 1142, dtype: int64


In [204]:
# Preview the first 10 rows
print(df_events.head(10))

   case_id             activity           timestamp               resource  \
0     1000             Check-in 2026-06-01 21:59:00       Front Desk Staff   
1     1000               Triage 2026-06-01 22:06:00            Nurse Sarah   
2     1000  Doctor Consultation 2026-06-01 22:58:00              Dr. Smith   
3     1000                X-Ray 2026-06-01 23:21:00              Tech Mike   
4     1000               Triage 2026-06-01 23:37:00            Nurse Sarah   
5     1000  Doctor Consultation 2026-06-01 23:47:00              Dr. Smith   
6     1000            Discharge 2026-06-02 00:04:00  Discharge Coordinator   
7     1001             Check-in 2026-06-03 17:16:00       Front Desk Staff   
8     1001               Triage 2026-06-03 17:28:00            Nurse Sarah   
9     1001  Doctor Consultation 2026-06-03 18:11:00               Dr. Chen   

             location acuity_level  patient_age patient_gender  \
0      Reception Desk  ESI Level 3           32           Male   
1       T

In [205]:
# Preview the last 10 rows
print(df_events.tail(10))

      case_id             activity           timestamp               resource  \
1132     1198             Check-in 2026-06-02 05:33:00       Front Desk Staff   
1133     1198               Triage 2026-06-02 05:51:00            Nurse Sarah   
1134     1198  Doctor Consultation 2026-06-02 06:47:00              Dr. House   
1135     1198                X-Ray 2026-06-02 07:12:00              Tech Mike   
1136     1198            Discharge 2026-06-02 07:42:00  Discharge Coordinator   
1137     1199             Check-in 2026-06-01 20:12:00       Front Desk Staff   
1138     1199               Triage 2026-06-01 20:24:00            Nurse Sarah   
1139     1199  Doctor Consultation 2026-06-01 21:13:00              Dr. Smith   
1140     1199                X-Ray 2026-06-01 21:24:00              Tech Mike   
1141     1199            Discharge 2026-06-01 21:44:00  Discharge Coordinator   

                location acuity_level  patient_age patient_gender  \
1132      Reception Desk  ESI Level 2  

In [206]:
# Preview the output
display(df_events.head(15))

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
5,1000,Doctor Consultation,2026-06-01 23:47:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,False
6,1000,Discharge,2026-06-02 00:04:00,Discharge Coordinator,Exit Lobby,ESI Level 3,32,Male,Dr. Smith,True
7,1001,Check-in,2026-06-03 17:16:00,Front Desk Staff,Reception Desk,ESI Level 2,45,Male,Dr. Chen,False
8,1001,Triage,2026-06-03 17:28:00,Nurse Sarah,Triage Room 1,ESI Level 2,45,Male,Dr. Chen,False
9,1001,Doctor Consultation,2026-06-03 18:11:00,Dr. Chen,Consultation Bay B,ESI Level 2,45,Male,Dr. Chen,False


In [207]:
# Save to CSV for Task 1 BigQuery ingestion
df_events.to_csv("careflow_synthetic_ehr_logs.csv", index=False)
print(
    "\nDataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'."
)


Dataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'.


Loading the Dataset

In [208]:
#  Load the raw dataset
df = pd.read_csv("careflow_synthetic_ehr_logs.csv")
df

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
...,...,...,...,...,...,...,...,...,...,...
1137,1199,Check-in,2026-06-01 20:12:00,Front Desk Staff,Reception Desk,ESI Level 4,43,Female,Dr. Smith,False
1138,1199,Triage,2026-06-01 20:24:00,Nurse Sarah,Triage Room 1,ESI Level 4,43,Female,Dr. Smith,False
1139,1199,Doctor Consultation,2026-06-01 21:13:00,Dr. Smith,Consultation Bay B,ESI Level 4,43,Female,Dr. Smith,False
1140,1199,X-Ray,2026-06-01 21:24:00,Tech Mike,Radiology Lab 2,ESI Level 4,43,Female,Dr. Smith,False


Dataset Description

In [209]:
df.shape

(1142, 10)

In [210]:
print(f"Original row count: {len(df)}")

Original row count: 1142


Attributes of the dataset

In [211]:
df.columns

Index(['case_id', 'activity', 'timestamp', 'resource', 'location',
       'acuity_level', 'patient_age', 'patient_gender', 'attending_doctor',
       'paperwork_missing_flag'],
      dtype='object')

Checking the consistency of the dataset.

First Five Rows

In [212]:
df.head()

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True


Last Five Rows



In [213]:
df.tail()

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
1137,1199,Check-in,2026-06-01 20:12:00,Front Desk Staff,Reception Desk,ESI Level 4,43,Female,Dr. Smith,False
1138,1199,Triage,2026-06-01 20:24:00,Nurse Sarah,Triage Room 1,ESI Level 4,43,Female,Dr. Smith,False
1139,1199,Doctor Consultation,2026-06-01 21:13:00,Dr. Smith,Consultation Bay B,ESI Level 4,43,Female,Dr. Smith,False
1140,1199,X-Ray,2026-06-01 21:24:00,Tech Mike,Radiology Lab 2,ESI Level 4,43,Female,Dr. Smith,False
1141,1199,Discharge,2026-06-01 21:44:00,Discharge Coordinator,Exit Lobby,ESI Level 4,43,Female,Dr. Smith,False


Overview of the Dataset

In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   case_id                 1142 non-null   int64 
 1   activity                1142 non-null   object
 2   timestamp               1142 non-null   object
 3   resource                1142 non-null   object
 4   location                1142 non-null   object
 5   acuity_level            1142 non-null   object
 6   patient_age             1142 non-null   int64 
 7   patient_gender          1142 non-null   object
 8   attending_doctor        1142 non-null   object
 9   paperwork_missing_flag  1142 non-null   bool  
dtypes: bool(1), int64(2), object(7)
memory usage: 81.5+ KB


Summary Statistics of the Numerical Features

In [215]:
df.describe()

,case_id,patient_age
count,1142.000000,1142.000000
mean,1099.119089,54.056918
std,58.401912,19.220737
min,1000.000000,18.000000
25%,1047.250000,37.000000
50%,1099.000000,56.000000
75%,1150.000000,72.000000
max,1199.000000,85.000000


Data Preprocessing

Data Cleaning Phase


Checking for Null Values

In [216]:
# 1. Drop rows with missing critical process columns
df = df.dropna(subset=["case_id", "activity", "timestamp"])

In [217]:
df.isnull().sum()

,0
case_id,0
activity,0
timestamp,0
resource,0
location,0
acuity_level,0
patient_age,0
patient_gender,0
attending_doctor,0
paperwork_missing_flag,0


Datatype of each columns

In [218]:
df.dtypes

,0
case_id,int64
activity,object
timestamp,object
resource,object
location,object
acuity_level,object
patient_age,int64
patient_gender,object
attending_doctor,object
paperwork_missing_flag,bool


In [219]:
# 2. Convert timestamp to actual datetime objects
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [220]:
df.dtypes

,0
case_id,int64
activity,object
timestamp,datetime64[ns]
resource,object
location,object
acuity_level,object
patient_age,int64
patient_gender,object
attending_doctor,object
paperwork_missing_flag,bool


In [221]:
# 3. Standardize text strings (strip whitespace, capitalize properly)
df["activity"] = df["activity"].str.strip()
df["resource"] = df["resource"].str.strip()
df["location"] = df["location"].str.strip()

In [222]:
# 4. Remove exact duplicate events
df = df.drop_duplicates()

In [223]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 0


In [224]:
# 5. Sort chronologically per case ID
df = df.sort_values(by=["case_id", "timestamp"]).reset_index(drop=True)

In [225]:
# 6. Save the cleaned dataset
df.to_csv("careflow_cleaned_ehr_logs.csv", index=False)
print(f"Cleaned row count: {len(df)}")
print("Dataset cleaned successfully and saved as 'careflow_cleaned_ehr_logs.csv'!")

Cleaned row count: 1142
Dataset cleaned successfully and saved as 'careflow_cleaned_ehr_logs.csv'!


Separate Columns for Date and Time derived from the Timestamp Column

In [226]:
df['date'] = df['timestamp'].dt.date
df['time'] = df['timestamp'].dt.time
print(df)

      case_id             activity           timestamp               resource  \
0        1000             Check-in 2026-06-01 21:59:00       Front Desk Staff   
1        1000               Triage 2026-06-01 22:06:00            Nurse Sarah   
2        1000  Doctor Consultation 2026-06-01 22:58:00              Dr. Smith   
3        1000                X-Ray 2026-06-01 23:21:00              Tech Mike   
4        1000               Triage 2026-06-01 23:37:00            Nurse Sarah   
...       ...                  ...                 ...                    ...   
1137     1199             Check-in 2026-06-01 20:12:00       Front Desk Staff   
1138     1199               Triage 2026-06-01 20:24:00            Nurse Sarah   
1139     1199  Doctor Consultation 2026-06-01 21:13:00              Dr. Smith   
1140     1199                X-Ray 2026-06-01 21:24:00              Tech Mike   
1141     1199            Discharge 2026-06-01 21:44:00  Discharge Coordinator   

                location ac

In [227]:
# Preview the output
display(df.head(15))

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag,date,time
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,21:59:00
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,22:06:00
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,22:58:00
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,23:21:00
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,23:37:00
5,1000,Doctor Consultation,2026-06-01 23:47:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,False,2026-06-01,23:47:00
6,1000,Discharge,2026-06-02 00:04:00,Discharge Coordinator,Exit Lobby,ESI Level 3,32,Male,Dr. Smith,True,2026-06-02,00:04:00
7,1001,Check-in,2026-06-03 17:16:00,Front Desk Staff,Reception Desk,ESI Level 2,45,Male,Dr. Chen,False,2026-06-03,17:16:00
8,1001,Triage,2026-06-03 17:28:00,Nurse Sarah,Triage Room 1,ESI Level 2,45,Male,Dr. Chen,False,2026-06-03,17:28:00
9,1001,Doctor Consultation,2026-06-03 18:11:00,Dr. Chen,Consultation Bay B,ESI Level 2,45,Male,Dr. Chen,False,2026-06-03,18:11:00


Let’s take a quick look at the frequency of each unique value in every column of the dataset using their value counts.

In [228]:
df['case_id'].value_counts()

,count
case_id,
1000,7
1002,7
1005,7
1004,7
1006,7
...,...
1195,5
1196,5
1197,5


In [229]:
df['activity'].value_counts()

,count
activity,
Triage,271
Doctor Consultation,271
Check-in,200
X-Ray,200
Discharge,200


In [230]:
df['resource'].value_counts()

,count
resource,
Nurse Sarah,271
Front Desk Staff,200
Tech Mike,200
Discharge Coordinator,200
Dr. Smith,65
Dr. Adams,63
Dr. House,52
Dr. Chen,48
Dr. Robert,43


In [231]:
df['location'].value_counts()

,count
location,
Triage Room 1,271
Consultation Bay B,271
Reception Desk,200
Radiology Lab 2,200
Exit Lobby,200


In [232]:
df['acuity_level'].value_counts()

,count
acuity_level,
ESI Level 3,300
ESI Level 4,298
ESI Level 5,283
ESI Level 2,261


In [233]:
df['patient_age'].value_counts()

,count
patient_age,
78,51
72,41
54,37
28,37
32,32
...,...
50,5
25,5
53,5


In [234]:
df['patient_gender'].value_counts()

,count
patient_gender,
Male,417
Female,377
Other,348


In [235]:
df['attending_doctor'].value_counts()

,count
attending_doctor,
Dr. Smith,268
Dr. Adams,258
Dr. House,221
Dr. Chen,216
Dr. Robert,179


In [236]:
df['paperwork_missing_flag'].value_counts()

,count
paperwork_missing_flag,
False,716
True,426


In [237]:
df['date'].value_counts()

,count
date,
2026-06-02,445
2026-06-03,360
2026-06-01,214
2026-06-04,123


Filtered Data

A summary table highlighting most missing paperwork:

In [238]:
filtered_df = df[df['paperwork_missing_flag'] == True]
display(filtered_df.head())

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag,date,time
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,21:59:00
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,22:06:00
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,22:58:00
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,23:21:00
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True,2026-06-01,23:37:00


Aggregated Dataset:

1. Patient Case-Level Summary (Duration & Activity Count):
Summarizes each patient's entire visit into a single row.

In [239]:
case_summary = (
    df.groupby("case_id")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        total_steps=("activity", "count"),
        acuity=("acuity_level", "first"),
        doctor=("attending_doctor", "first"),
    )
    .reset_index()
)
case_summary["total_duration"] = (
    case_summary["end_time"] - case_summary["start_time"]
)
display(case_summary.head())

,case_id,start_time,end_time,total_steps,acuity,doctor,total_duration
0,1000,2026-06-01 21:59:00,2026-06-02 00:04:00,7,ESI Level 3,Dr. Smith,0 days 02:05:00
1,1001,2026-06-03 17:16:00,2026-06-03 19:02:00,5,ESI Level 2,Dr. Chen,0 days 01:46:00
2,1002,2026-06-02 13:23:00,2026-06-02 15:17:00,7,ESI Level 4,Dr. Adams,0 days 01:54:00
3,1003,2026-06-02 01:02:00,2026-06-02 02:30:00,5,ESI Level 2,Dr. Smith,0 days 01:28:00
4,1004,2026-06-02 15:06:00,2026-06-02 17:00:00,7,ESI Level 2,Dr. House,0 days 01:54:00


2. Resource & Staff Workload Aggregation:
Measures how busy each staff member or resource is and their error/missing paperwork frequency.

In [240]:
resource_workload = (
    df.groupby("resource")
    .agg(
        total_tasks=("activity", "count"),
        unique_patients=("case_id", "nunique"),
        missing_paperwork_count=("paperwork_missing_flag", "sum"),
    )
    .reset_index()
)
display(resource_workload.head())

,resource,total_tasks,unique_patients,missing_paperwork_count
0,Discharge Coordinator,200,200,71
1,Dr. Adams,63,44,19
2,Dr. Chen,48,40,8
3,Dr. House,52,39,13
4,Dr. Robert,43,31,12


3. Acuity Level & Demographics Breakdown:
Analyzes patient demographics and volume based on emergency severity (ESI Levels).

In [241]:
acuity_stats = (
    df.groupby("acuity_level")
    .agg(
        total_cases=("case_id", "nunique"),
        avg_age=("patient_age", "mean"),
        missing_paperwork=("paperwork_missing_flag", "sum"),
    )
    .reset_index()
)
display(acuity_stats.head())

,acuity_level,total_cases,avg_age,missing_paperwork
0,ESI Level 2,45,57.245211,108
1,ESI Level 3,52,51.643333,120
2,ESI Level 4,54,56.570470,84
3,ESI Level 5,49,51.028269,114


4. Activity & Location Bottleneck Aggregation:
Identifies which hospital rooms or activities handle the highest volume and where paperwork issues typically happen.

In [242]:
location_bottlenecks = (
    df.groupby(["location", "activity"])
    .agg(
        event_count=("activity", "count"),
        missing_paperwork_rate=("paperwork_missing_flag", "mean"),
    )
    .reset_index()
)
# Convert rate to percentage
location_bottlenecks["missing_paperwork_rate"] = (
    location_bottlenecks["missing_paperwork_rate"] * 100
)
display(location_bottlenecks.head())

,location,activity,event_count,missing_paperwork_rate
0,Consultation Bay B,Doctor Consultation,271,26.199262
1,Exit Lobby,Discharge,200,35.500000
2,Radiology Lab 2,X-Ray,200,35.500000
3,Reception Desk,Check-in,200,35.500000
4,Triage Room 1,Triage,271,52.398524
